In [8]:
import pandas as pd
import numpy as np
import pennylane as qml
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import root_mean_squared_error, r2_score
from qiskit.circuit.library import pauli_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

import os


os.makedirs("../results/experiment_16", exist_ok=True)

In [9]:
dataset = pd.read_csv("../dataset/riemann_features.csv")

X = dataset.drop(columns=["distance"])
y = dataset["distance"]

X = X[:2000]
y = y[:2000]

split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

features = pd.read_csv("../results/data_analysis/selected_features.csv")["feature"].tolist()

# Clássico

In [10]:
def run_classical_experiment(features):
    results = []

    n_features = [i for i in range(1, 11)]

    for n in n_features:
        print(f"Running classical experiment with {n} features using Pauli feature map")

        X_train_subset = X_train[features[:n]].to_numpy()
        X_test_subset = X_test[features[:n]].to_numpy()

        scaler_X = MinMaxScaler(feature_range=(0, np.pi))
        X_train_scaled = scaler_X.fit_transform(X_train_subset)
        X_test_scaled = scaler_X.transform(X_test_subset)

        svr = SVR(kernel="rbf", C=10, epsilon=0.01)
        svr.fit(X_train_scaled, y_train)
        y_pred = svr.predict(X_test_scaled)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            "N Features": n,
            "RMSE": rmse,
            "R2": r2,
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)


In [11]:
results = run_classical_experiment(features)
print(results)
results.to_csv("../results/experiment_16/classical_results.csv", index=False)

Running classical experiment with 1 features using Pauli feature map
Running classical experiment with 2 features using Pauli feature map
Running classical experiment with 3 features using Pauli feature map
Running classical experiment with 4 features using Pauli feature map
Running classical experiment with 5 features using Pauli feature map
Running classical experiment with 6 features using Pauli feature map
Running classical experiment with 7 features using Pauli feature map
Running classical experiment with 8 features using Pauli feature map
Running classical experiment with 9 features using Pauli feature map
Running classical experiment with 10 features using Pauli feature map
   N Features      RMSE        R2
0           9  0.056229  0.961104
1           6  0.064620  0.948630
2          10  0.067752  0.943530
3           8  0.071701  0.936754
4           7  0.072381  0.935549
5           5  0.075013  0.930776
6           3  0.096353  0.885787
7           4  0.111433  0.847241
8  

# Quantum

## PauliFeatureMap

In [12]:
def riemann_advanced_data_map(x):
    x = np.asarray(x)

    x_safe = x + 1e-8

    theta_like = np.sum(x_safe * np.log(x_safe))
    val = np.cos(theta_like)

    for i in range(len(x) - 1):
        val += 0.3 * np.cos(x[i] - x[i+1])
        val += 0.3 * np.sin(x[i] * x[i+1])

    return val


def riemann_data_map(x):
    x = np.asarray(x)
    if len(x) == 1:
        return x[0]
    x_safe = x + 1e-8
    return np.sum(x_safe * np.log(x_safe))


def create_quantum_kernel(n_qubits, reps=3, entanglement="full", paulis=["ZZ", "Z"]):
    feature_map = pauli_feature_map(
        feature_dimension=n_qubits,
        reps=reps,
        entanglement=entanglement,
        paulis=paulis,
        data_map_func=riemann_advanced_data_map    
    )
    sampler = StatevectorSampler()
    fidelity = ComputeUncompute(sampler=sampler)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)
    return quantum_kernel


def run_quantum_experiment_pauli_feature_map():
    results = []
    n_features = [6, 9, 10]

    for n in n_features:
        print(f"Running quantum experiment with {n} features using Pauli feature map")

        X_train_subset = X_train[features[:n]].to_numpy()
        X_test_subset = X_test[features[:n]].to_numpy()

        scaler_X = MinMaxScaler(feature_range=(0, np.pi))
        X_train_scaled = scaler_X.fit_transform(X_train_subset)
        X_test_scaled = scaler_X.transform(X_test_subset)

        quantum_kernel = create_quantum_kernel(n_qubits=n, reps=2, entanglement="full", paulis=["Y", "YY"])

        svr = SVR(kernel=quantum_kernel.evaluate, C=10, epsilon=0.01)
        svr.fit(X_train_scaled, y_train)
        y_pred = svr.predict(X_test_scaled)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        results.append({
            "N Features": n,
            "RMSE": rmse,
            "R2": r2,
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

In [13]:
# results_pauli = run_quantum_experiment_pauli_feature_map()
# print(results_pauli)
# results_pauli.to_csv("../results/experiment_16/quantum_results_pauli.csv", index=False)

## Angle Encoding

In [14]:
from mpmath import grampoint, siegeltheta, siegelz
import numpy as np
import pennylane as qml


def _gram_weights(n_qubits):
    points = [float(grampoint(n)) for n in range(n_qubits + 1)]
    return np.diff(points) % (2 * np.pi)


def _theta_weights(n_qubits, t_min=14.0, t_max=50.0):
    t_grid = np.linspace(t_min, t_max, n_qubits)
    theta = np.array([float(siegeltheta(t)) for t in t_grid])
    return np.gradient(theta) % (2 * np.pi)


def _hardy_weights(n_qubits, t_min=14.0, t_max=50.0):
    t_grid = np.linspace(t_min, t_max, n_qubits)
    z = np.array([float(siegelz(t)) for t in t_grid])
    return 2 * np.pi * (z - z.min()) / (z.max() - z.min())


_ROTATION_MAP = {
    "gram":  (qml.RX, _gram_weights),
    "theta": (qml.RY, _theta_weights),
    "hardy": (qml.RZ, _hardy_weights),
}


def make_kernel_v2(
    n_qubits,
    rotation_first="X",
    rotation_second="Y",
    sources=("gram", "theta", "hardy"),
):
    layers = []
    for src in sources:
        if src not in _ROTATION_MAP:
            raise ValueError(f"Fonte desconhecida '{src}'. Opções: {list(_ROTATION_MAP)}")
        gate, weight_fn = _ROTATION_MAP[src]
        layers.append((gate, weight_fn(n_qubits)))

    dev = qml.device("lightning.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def kernel_fn(x1, x2):
        qml.AngleEmbedding(x1, wires=range(n_qubits), rotation=rotation_first)

        for gate, weights in layers:
            for i in range(n_qubits):
                gate(weights[i], wires=i)
            for i in range(n_qubits):
                qml.CNOT(wires=[i, (i + 1) % n_qubits])

        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation=rotation_second)
        return qml.expval(qml.Projector([0] * n_qubits, wires=range(n_qubits)))

    return kernel_fn


def make_kernel(n_qubits, rotation_first="X", rotation_second="Y"):
    dev = qml.device("lightning.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def kernel_fn(x1, x2):
        qml.AngleEmbedding(x1, wires=range(n_qubits), rotation=rotation_first)
        qml.BasicEntanglerLayers(
            weights=np.zeros((1, n_qubits)),
            wires=range(n_qubits),
            rotation=qml.RZ
        )
        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation=rotation_second)
        return qml.expval(qml.Projector([0] * n_qubits, wires=range(n_qubits)))

    return kernel_fn


def run_quantum_experiment_angle_encoding(features, rotation_first="X", rotation_second="Y"):
    results = []
    n_features = [6, 9, 10]

    for n in n_features:
        print(f"Running quantum experiment with {n} features and rotations {rotation_first} and {rotation_second}")

        X_train_subset = X_train[features[:n]].to_numpy()
        X_test_subset = X_test[features[:n]].to_numpy()

        scaler_X = MinMaxScaler(feature_range=(0, np.pi))
        X_train_scaled = scaler_X.fit_transform(X_train_subset)
        X_test_scaled = scaler_X.transform(X_test_subset)

        kernel_fn = make_kernel(n, rotation_first, rotation_second)
        kernel_mat = lambda A, B, kf=kernel_fn: qml.kernels.kernel_matrix(A, B, kf)

        svr = SVR(kernel=kernel_mat, C=10, epsilon=0.01)
        svr.fit(X_train_scaled, y_train)
        y_pred = svr.predict(X_test_scaled)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        print(f"RMSE: {rmse}, R2: {r2}\n")

        results.append({
            "N Features": n,
            "RMSE": rmse,
            "R2": r2,
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)


def run_quantum_experiment_angle_encoding_v2(features, rotation_first="X", rotation_second="Y"):
    results = []
    n_features = [6, 9, 10]

    for n in n_features:
        print(f"Running quantum experiment with {n} features and rotations {rotation_first} and {rotation_second}")

        X_train_subset = X_train[features[:n]].to_numpy()
        X_test_subset = X_test[features[:n]].to_numpy()

        scaler_X = MinMaxScaler(feature_range=(0, np.pi))
        X_train_scaled = scaler_X.fit_transform(X_train_subset)
        X_test_scaled = scaler_X.transform(X_test_subset)

        kernel_fn = make_kernel_v2(n, rotation_first, rotation_second)
        kernel_mat = lambda A, B, kf=kernel_fn: qml.kernels.kernel_matrix(A, B, kf)

        svr = SVR(kernel=kernel_mat, C=10, epsilon=0.01)
        svr.fit(X_train_scaled, y_train)
        y_pred = svr.predict(X_test_scaled)

        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        print(f"RMSE: {rmse}, R2: {r2}\n")

        results.append({
            "N Features": n,
            "RMSE": rmse,
            "R2": r2,
        })

    return pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

In [16]:
from multiprocessing import Process


def run_YY(features):
    results_YY = run_quantum_experiment_angle_encoding(features, rotation_first="Y", rotation_second="Y")
    print(results_YY)
    results_YY.to_csv("../results/experiment_16/quantum_results_YY.csv", index=False)


def run_XY(features):
    results_XY = run_quantum_experiment_angle_encoding(features, rotation_first="X", rotation_second="Y")
    print(results_XY)
    results_XY.to_csv("../results/experiment_16/quantum_results_XY.csv", index=False)


def run_XY_v2(features):
    results_XY_v2 = run_quantum_experiment_angle_encoding_v2(features, rotation_first="X", rotation_second="Y")
    print(results_XY_v2)
    results_XY_v2.to_csv("../results/experiment_16/quantum_results_XY_v2.csv", index=False)


def run_YY_v2(features):
    results_YY_v2 = run_quantum_experiment_angle_encoding_v2(features, rotation_first="Y", rotation_second="Y")
    print(results_YY_v2)
    results_YY_v2.to_csv("../results/experiment_16/quantum_results_YY_v2.csv", index=False)


p1 = Process(target=run_YY, args=(features,))
p2 = Process(target=run_XY, args=(features,))
p3 = Process(target=run_XY_v2, args=(features,))
p4 = Process(target=run_YY_v2, args=(features,))

p1.start()
p2.start()
p3.start()
p4.start()
p1.join()
p2.join()
p3.join()
p4.join()

Running quantum experiment with 6 features and rotations Y and Y


Running quantum experiment with 6 features and rotations X and Y
Running quantum experiment with 6 features and rotations X and Y
Running quantum experiment with 6 features and rotations Y and Y
RMSE: 45.81705810710515, R2: -25823.63023672537

Running quantum experiment with 9 features and rotations Y and Y


KeyboardInterrupt: 